In [ ]:
# AI-Based Hotel Room Pricing Prediction 2025
# Advanced Algorithm – No Room IDs, Profit Guaranteed
# ============================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib

# ===============================
# 1️⃣ Load CSV files
# ===============================
booking = pd.read_csv(r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\booking_history.csv')       # Historical booking
revenue = pd.read_csv(r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\revenue.csv')               # Historical revenue
events = pd.read_csv(r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\events.csv')                 # Event data for 2025
competitor = pd.read_csv(r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\competitor.csv')         # Competitor prices
current_price = pd.read_csv(r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\actual_room_price.csv')  # Actual 2025 room prices
# Map months to numbers
month_map = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}

for df in [booking, revenue, events, competitor]:
    df['Month_Num'] = df['Month'].map(month_map)

# ===============================
# 2️⃣ Merge historical booking & revenue
# ===============================
df_2024 = pd.merge(booking, revenue, on=['Month','Room_Type','year','Month_Num'], how='left')
df_2024['Occupancy_Factor'] = df_2024['Booked_Rooms'] / df_2024['Total_Rooms']
df_2024['Revenue_per_Room_Calc'] = df_2024['Monthly_Revenue'] / df_2024['Booked_Rooms']

# Encode Room_Type
le_room = LabelEncoder()
df_2024['Room_Type_Code'] = le_room.fit_transform(df_2024['Room_Type'])
current_price['Room_Type_Code'] = le_room.transform(current_price['Room_Type'])

# ===============================
# 3️⃣ Event impact & competitor average price
# ===============================
event_impact = events.groupby('Month_Num').size().reset_index(name='Event_Count')
comp_avg = competitor.groupby('Month_Num').agg({'Room_price':'mean'}).reset_index()

# ===============================
# 4️⃣ Prepare training features
# ===============================
df_features = pd.merge(df_2024, event_impact, on='Month_Num', how='left')
df_features = pd.merge(df_features, comp_avg, on='Month_Num', how='left')
df_features = pd.merge(df_features, current_price[['Room_Type_Code','Room_per_night_price']], on='Room_Type_Code', how='left')
df_features.fillna(0, inplace=True)

feature_cols = ['Month_Num','Room_Type_Code','Occupancy_Factor','Revenue_per_Room_Calc','Event_Count','Room_price','Room_per_night_price']
X = df_features[feature_cols]
y = df_features['Room_per_night_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ===============================
# 5️⃣ Train Random Forest Regressor
# ===============================
model = RandomForestRegressor(n_estimators=250, max_depth=8, random_state=42)
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print("Model trained. MSE:", mse)

# ===============================
# 6️⃣ Predict 2025 room prices
# ===============================
months = list(range(1, 13))
room_types = df_features['Room_Type_Code'].unique()
predict_data = []

for m in months:
    for rt in room_types:
        # Event count
        event_count = int(event_impact[event_impact['Month_Num']==m]['Event_Count'].values[0]) if m in event_impact['Month_Num'].values else 0
        # Competitor average price
        comp_price = float(comp_avg[comp_avg['Month_Num']==m]['Room_price'].values[0]) if m in comp_avg['Month_Num'].values else 0
        # Current 2025 price
        current = float(current_price[current_price['Room_Type_Code']==rt]['Room_per_night_price'].values[0])
        predict_data.append([m, rt, event_count, comp_price, current])

predict_df = pd.DataFrame(predict_data, columns=['Month_Num','Room_Type_Code','Event_Count','Room_price','Room_per_night_price'])
predict_df['Occupancy_Factor'] = 0.85          # optimistic occupancy assumption
predict_df['Revenue_per_Room_Calc'] = 0
predict_df.fillna(0, inplace=True)

X_pred = predict_df[feature_cols]
predict_df['Predicted_Room_Price'] = model.predict(X_pred)

# ===============================
# 7️⃣ Ensure all predicted prices are above actual prices
# ===============================
min_increase_pct = 0.05  # 5% minimum increase
max_increase_pct = 0.15  # 15% maximum increase

predict_df['Predicted_Room_Price'] = predict_df.apply(
    lambda row: max(
        row['Predicted_Room_Price'], 
        row['Room_per_night_price'] * (1 + np.random.uniform(min_increase_pct, max_increase_pct))
    ),
    axis=1
)

# Map back Room_Type and Month names
predict_df['Room_Type'] = le_room.inverse_transform(predict_df['Room_Type_Code'].astype(int))
month_reverse = {v:k for k,v in month_map.items()}
predict_df['Month'] = predict_df['Month_Num'].map(month_reverse)

# ===============================
# 8️⃣ Create final CSV (No Room IDs)
# ===============================
final_df = predict_df[['Month','Room_Type','Predicted_Room_Price']].copy()
final_df.rename(columns={'Predicted_Room_Price':'Room_per_night_price'}, inplace=True)
final_df.to_csv(r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\model\predicted_room_prices_2025.csv', index=False)

# Save trained model
joblib.dump(model, r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\model\rf_room_pricing_model.joblib')

print("Predicted 2025 room prices (no Room IDs) saved successfully!")



✅ Model trained. MSE: 0.0
✅ Predicted 2025 room prices (no Room IDs) saved successfully!
